# Grouped 3-Model x 3-Seed Experiments

This notebook replaces the original random filename split. It creates `grouped_v1_contiguous_24`, removes exact clear+turbid pair duplicates, and runs the default U-Net, parameter-matched U-Net, and residual backbone with seeds 42, 123, and 2026.

Select a GPU runtime before starting. If Colab disconnects, reconnect and run all cells again. Completed jobs are skipped and partial jobs resume from `latest.pth`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before training.'
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = 'https://github.com/bedirhanozturk1/Underwater_Resnet_Restoration.git'
DRIVE_ROOT = '/content/drive/MyDrive/underwater_resnet_project'
!test -d /content/Underwater_Resnet_Restoration/.git || git clone {REPO_URL} /content/Underwater_Resnet_Restoration
%cd /content/Underwater_Resnet_Restoration
!git pull --ff-only
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
required = [
    Path(DRIVE_ROOT) / 'datasets/clear_underwater_color_patch/canon_patch',
    Path(DRIVE_ROOT) / 'datasets/turbidty_underwater_color_patch',
]
missing = [str(path) for path in required if not path.is_dir()]
assert not missing, f'Missing dataset directories: {missing}'
print('Dataset directories found.')

## Run or Resume the Full Matrix

This is the only long-running cell. Outputs are written directly to Google Drive under `experiments/grouped_v1`, so they survive runtime disconnections.

In [ ]:
!python scripts/run_grouped_experiments.py \
  --drive-root {DRIVE_ROOT} \
  --seeds 42 123 2026 \
  --evaluation-seed 2026 \
  --epochs 50 \
  --batch-size 16 \
  --num-workers 2

In [ ]:
import pandas as pd
summary = Path(DRIVE_ROOT) / 'experiments/grouped_v1/summaries/metric_summary_mean_std.csv'
assert summary.exists(), 'The full matrix has not completed yet. Rerun the training cell to resume.'
pd.read_csv(summary)